In [33]:
import pandas as pd

In [34]:
df = pd.read_csv('data_final.csv', delimiter='%')
df.head()

,inn,address,main_okved,main_descr,add_okveds,add_descrs
0,7708503727,"107174, г. Москва, вн. тер. г. муниципальный о...",49.20,Деятельность железнодорожного транспорта: груз...,86.10|46.7|47.79|85.42|63.11|85.41.1|52.24|30....,Деятельность больничных организаций|Торговля о...
1,7736050003,"197229, г. Санкт-Петербург, вн. тер. г. муници...",46.71,"Торговля оптовая твердым, жидким и газообразны...",49.50.1|70.22|06.20|71.12.45|85.42.9|09.10|35....,Транспортирование по трубопроводам нефти и неф...
2,4716016979,"121353, г. Москва, вн. тер. г. муниципальный о...",35.12,Передача электроэнергии и технологическое прис...,61.10|35.30.2|35.30.3|49.4|35.13|85.42.9|56.29...,Деятельность в области связи на базе проводных...
3,7707049388,"191167, г. Санкт-Петербург, вн. тер. г. муници...",61.10,Деятельность в области связи на базе проводных...,42.21|62.02.1|62.02|62.02.4|71.11.1|68.1|77.39...,Строительство инженерных коммуникаций для водо...
4,7706061801,"123112, г. Москва, Пресненская набережная, д. ...",49.50.1,Транспортирование по трубопроводам нефти и неф...,63.99.1|52.10.21|71.20|72.19|49.50.11|49.50.12...,Деятельность по оказанию консультационных и ин...


In [35]:
import requests

In [36]:
with open("token1.txt", "r") as f:
    token1 = f.readline().strip()
with open("token2.txt", "r") as f:
    token2 = f.readline().strip()

In [37]:
url = 'https://api-fns.ru/api/multinfo'
api_out_f = open("api_out.csv", "w", buffering=1)
cur_token = token1
for inn in df["inn"]:
    try:
        comp_str = ''
        response = requests.get(
            url + '?req=' + str(inn) + '&key=' + cur_token
        )
        if response.status_code != 200:
            if cur_token == token1:
                cur_token = token2
            else:
                break
        body = response.json()['items'][0]['ЮЛ']
        comp_str += str(body['ИНН']) + '%'
        comp_str += str(body['КПП']) + '%'
        comp_str += str(body['ОГРН']) + '%'
        comp_str += str(body['ДатаОГРН']) + '%'
        comp_str += str(body['ДатаРег']) + '%'
        comp_str += str(body['Статус']) + '%'
        comp_str += str(body['Капитал']['СумКап']) + '%'
        comp_str += str(body['НаимПолнЮЛ']) + '\n'
        api_out_f.write(comp_str)
    except:
        continue
api_out_f.close()

In [38]:
api_df = pd.read_csv("api_out.csv", delimiter='%')
api_df.columns = ['inn', 'КПП', 'ОГРН', 'ДатаОГРН', 'ДатаРег', 'Статус', 'Капитал', 'НаимПолнЮЛ']
api_df.head()

,inn,КПП,ОГРН,ДатаОГРН,ДатаРег,Статус,Капитал,НаимПолнЮЛ
0,7736050003,781401001,1027700070518,2002-08-02,1993-02-25,Действующее,1.183676e+11,"ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО ""ГАЗПРОМ"""
1,4716016979,773101001,1024701893336,2002-08-20,2002-06-25,Действующее,1.149597e+12,"ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО ""ФЕДЕРАЛЬНАЯ СЕ..."
2,7707049388,784201001,1027700198767,2002-09-09,2002-09-09,Действующее,8.731408e+06,"ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО ""РОСТЕЛЕКОМ"""
3,7706061801,770301001,1027700049486,2002-07-24,1993-08-26,Действующее,7.249343e+06,"ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО ""ТРАНСНЕФТЬ"""
4,6315376946,502401001,1056315070350,2005-08-01,2005-08-01,Действующее,4.446704e+10,"ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО ""Т ПЛЮС"""


In [40]:
merged_df = pd.merge(df, api_df, on='inn')
merged_df.head()

,inn,address,main_okved,main_descr,add_okveds,add_descrs,КПП,ОГРН,ДатаОГРН,ДатаРег,Статус,Капитал,НаимПолнЮЛ
0,7736050003,"197229, г. Санкт-Петербург, вн. тер. г. муници...",46.71,"Торговля оптовая твердым, жидким и газообразны...",49.50.1|70.22|06.20|71.12.45|85.42.9|09.10|35....,Транспортирование по трубопроводам нефти и неф...,781401001,1027700070518,2002-08-02,1993-02-25,Действующее,1.183676e+11,"ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО ""ГАЗПРОМ"""
1,4716016979,"121353, г. Москва, вн. тер. г. муниципальный о...",35.12,Передача электроэнергии и технологическое прис...,61.10|35.30.2|35.30.3|49.4|35.13|85.42.9|56.29...,Деятельность в области связи на базе проводных...,773101001,1024701893336,2002-08-20,2002-06-25,Действующее,1.149597e+12,"ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО ""ФЕДЕРАЛЬНАЯ СЕ..."
2,7707049388,"191167, г. Санкт-Петербург, вн. тер. г. муници...",61.10,Деятельность в области связи на базе проводных...,42.21|62.02.1|62.02|62.02.4|71.11.1|68.1|77.39...,Строительство инженерных коммуникаций для водо...,784201001,1027700198767,2002-09-09,2002-09-09,Действующее,8.731408e+06,"ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО ""РОСТЕЛЕКОМ"""
3,7706061801,"123112, г. Москва, Пресненская набережная, д. ...",49.50.1,Транспортирование по трубопроводам нефти и неф...,63.99.1|52.10.21|71.20|72.19|49.50.11|49.50.12...,Деятельность по оказанию консультационных и ин...,770301001,1027700049486,2002-07-24,1993-08-26,Действующее,7.249343e+06,"ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО ""ТРАНСНЕФТЬ"""
4,6315376946,"143421, Московская область, г. о. Красногорск,...",35.11,Производство электроэнергии,49.3|71.12.45|86.90.4|56.29|71.12.12|68.10|35....,Деятельность прочего сухопутного пассажирского...,502401001,1056315070350,2005-08-01,2005-08-01,Действующее,4.446704e+10,"ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО ""Т ПЛЮС"""


In [41]:
merged_df.shape

(96, 13)